In [ ]:
!pip install -U langchain langchain-community langchain-openai faiss-cpu sentence-transformers

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 88.5/88.5 kB 3.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 506.7/506.7 kB 13.4 MB/s eta 0:00:00
  Attempting uninstall: langchain-core
    Found existing installation: langchain-core 1.2.19
    Uninstalling langchain-core-1.2.19:
      Successfully uninstalled langchain-core-1.2.19


In [ ]:
with open("/content/sample_data/cardata.txt", "w") as f:
    f.write(""" Maserati is an Italian luxury automobile manufacturer founded in 1914 in Bologna, Italy. The company was established by the Maserati brothers, including Alfieri Maserati, who played a key role in its early development.

Initially, Maserati focused on building race cars and quickly gained recognition in motorsports. In 1926, the company produced its first car, the Tipo 26, which debuted in racing and won its class.

The Maserati logo, a trident, was inspired by the statue of Neptune located in Bologna. It symbolizes strength, power, and vigor.

During the 1930s and 1950s, Maserati became one of the most successful racing brands. The company achieved major victories in Grand Prix racing. In 1957, Juan Manuel Fangio won the Formula One World Championship driving a Maserati 250F.

After its racing success, Maserati began producing road cars that combined luxury with high performance. Models such as the A6 series marked its entry into the consumer automobile market.

In 1968, Maserati was acquired by the French company Citroën. During this period, the company introduced technologically advanced cars but faced financial challenges.

In the 1970s, Maserati went through economic difficulties due to the global oil crisis. It was later taken over by Alejandro de Tomaso, who helped stabilize the brand.

In 1993, Maserati was acquired by the Italian automotive giant Fiat. Later, it became part of Ferrari, which improved its engineering quality and global reputation.

In recent years, Maserati has focused on luxury sports vehicles such as the Quattroporte, Ghibli, and Levante. The company has also entered the electric vehicle market with new innovations.

Today, Maserati is part of Stellantis and continues to produce high-performance luxury vehicles known for their design, speed, and exclusivity.""")

In [ ]:
!pip install langchain-groq

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 137.5/137.5 kB 4.1 MB/s eta 0:00:00


In [ ]:
from langchain_community.document_loaders import TextLoader

loader = TextLoader("/content/sample_data/cardata.txt")
documents = loader.load()

print(documents[0].page_content[:200])  # preview

 Maserati is an Italian luxury automobile manufacturer founded in 1914 in Bologna, Italy. The company was established by the Maserati brothers, including Alfieri Maserati, who played a key role in its


In [ ]:
from langchain_text_splitters import CharacterTextSplitter

text_splitter = CharacterTextSplitter(
    chunk_size=300,
    chunk_overlap=50
)

docs = text_splitter.split_documents(documents)

print("Total chunks:", len(docs))

Total chunks: 10


In [ ]:
from langchain_community.embeddings import HuggingFaceEmbeddings

embeddings = HuggingFaceEmbeddings(
    model_name="all-MiniLM-L6-v2"
)

/tmp/ipykernel_13469/4246038141.py:3: LangChainDeprecationWarning: The class `HuggingFaceEmbeddings` was deprecated in LangChain 0.2.2 and will be removed in 1.0. An updated version of the class exists in the `langchain-huggingface package and should be used instead. To use it run `pip install -U `langchain-huggingface` and import as `from `langchain_huggingface import HuggingFaceEmbeddings``.
  embeddings = HuggingFaceEmbeddings(
/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md: 0.00B [00:00, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

...

tokenizer_config.json:   0%|          | 0.00/350 [00:00<?, ?B/s]

vocab.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

In [ ]:
from langchain_community.vectorstores import FAISS

db = FAISS.from_documents(docs, embeddings)

In [ ]:
retriever = db.as_retriever(search_kwargs={"k": 3})

In [ ]:
def format_docs(docs):
    return "\n\n".join(doc.page_content for doc in docs)

In [ ]:
from google.colab import userdata
llm=userdata.get('GROKAPI')

In [ ]:
import os
os.environ["GROQ_API_KEY"] = llm

from langchain_groq import ChatGroq

llm = ChatGroq(
    model="llama-3.1-8b-instant"
)

In [ ]:
from langchain_core.runnables import RunnablePassthrough
from langchain_core.output_parsers import StrOutputParser
from langchain_core.prompts import ChatPromptTemplate

template = """Answer the question based only on the following context:
{context}

Question: {question}
"""
prompt = ChatPromptTemplate.from_template(template)

qa = (
    {
        "context": retriever | format_docs,   # ✅ HERE is where it's used
        "question": RunnablePassthrough()
    }
    | prompt
    | llm
    | StrOutputParser()
)

In [ ]:
qa.invoke("when founded the i need year?")

'1914.'